In [ ]:
import os
import json
from api import skraafoto
from PIL import Image

def fetch_image(item_id):
    data = skraafoto.fetch_skraafoto_item(item_id)

    img_url = data["assets"]["data"]["href"]
    img_data = skraafoto.get_image_from_url(img_url).content

    save_dir = f"skraafoto_images/{item_id}"
    os.makedirs(save_dir, exist_ok=True)

    img_path = os.path.join(save_dir, f"{item_id}.tif")
    with open(img_path, "wb") as f:
        f.write(img_data)

    bbox = data.get("bbox", [])
    metadata = {"item_id": item_id, "bbox": bbox, "crops": []}

    metadata_path = os.path.join(save_dir, "metadata.json")
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=4)

    print(f"Downloaded: {img_path} with metadata.")
    return img_path, metadata_path

def extract_crops(image_path, metadata_path, crop_size=1000, num_crops=10):
    image = Image.open(image_path)
    width, height = image.size

    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    crops_dir = os.path.join(os.path.dirname(image_path), "crops")
    os.makedirs(crops_dir, exist_ok=True)

    max_cols, max_rows = width // crop_size, height // crop_size
    total_possible_crops = max_cols * max_rows
    num_crops = min(num_crops, total_possible_crops)

    crop_positions = [(col * crop_size, row * crop_size) for row in range(max_rows) for col in range(max_cols)][:num_crops]
    metadata["crops"] = []

    for i, (x, y) in enumerate(crop_positions):
        crop_filename = os.path.join(crops_dir, f"crop_{i+1}.jpg")
        cropped_image = image.crop((x, y, x + crop_size, y + crop_size))
        cropped_image.save(crop_filename)
        metadata["crops"].append({
            "crop_id": i+1,
            "crop_filename": crop_filename,
            "bbox": {"x_min": x, "y_min": y, "x_max": x + crop_size, "y_max": y + crop_size}
        })

    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=4)

    print(f"{num_crops} cropped images saved successfully.")

def extract_crops_for_all_images(crop_size=1000, num_crops=10):
    base_dir = "skraafoto_images"
    for item_id in os.listdir(base_dir):
        item_dir = os.path.join(base_dir, item_id)
        if os.path.isdir(item_dir):
            image_path = os.path.join(item_dir, f"{item_id}.tif")
            metadata_path = os.path.join(item_dir, "metadata.json")
            if os.path.exists(image_path) and os.path.exists(metadata_path):
                extract_crops(image_path, metadata_path, crop_size, num_crops)
            else:
                print(f"Skipping {item_id} (missing image or metadata).")

item_ids = ["2023_82_24_2_0011_00000400", "2023_82_24_2_0011_00000401"]
for item_id in item_ids:
    fetch_image(item_id)
extract_crops_for_all_images()


Downloaded: skraafoto_images/2023_82_24_2_0011_00000400/2023_82_24_2_0011_00000400.tif with metadata including bbox.
Downloaded: skraafoto_images/2023_82_24_2_0011_00000401/2023_82_24_2_0011_00000401.tif with metadata including bbox.
10 cropped images saved successfully with bounding box metadata.
10 cropped images saved successfully with bounding box metadata.
